In [3]:
# 检查 AMD GPU 是否可用
import torch
print(f'ROCm 可用: {torch.cuda.is_available()}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "未检测到"}')

^C


In [ ]:
!pip install PyPDF2 -q

In [ ]:
import requests
import time
import concurrent.futures


class TextAnalyzer:
    def __init__(self, model_name="qwen3-coder:30b", api_url="http://open-webui-ollama.open-webui:11434"):
        self.model_name = model_name
        self.api_url = f"{api_url}/api/chat"
        self.headers = {"Content-Type": "application/json"}
        self.timeout = 600

    def _call_llm(self, prompt, temperature=0.3, max_retries=3):
        """底层 API 调用函数 """

        # 1. 输入合法性检查：拦截空输入
        if not prompt or not str(prompt).strip():
            return "Error: 检测到空输入或非法字符，已拦截。"

        payload = {
            "model": self.model_name,
            "messages": [
                # System Role 设定：强化输出约束
                {"role": "system",
                 "content": "你是一个严谨的数据处理助手。请直接输出分析结果，绝对不要包含任何开场白、前缀或解释性废话。"},
                {"role": "user", "content": prompt}
            ],
            "stream": False,
            "options": {
                "num_ctx": 32768,
                "temperature": temperature
            }
        }

        # 2. 错误处理与重试机制 (API 生命周期管理)
        for attempt in range(max_retries):
            try:
                response = requests.post(self.api_url, json=payload, headers=self.headers, timeout=self.timeout)

                # 应对频率限制 线性退避策略
                if response.status_code == 429:
                    wait_time = 3 * (attempt + 1)
                    print(
                        f"    └── 触发 API 限流 (429)，等待 {wait_time}s 后重试 (尝试 {attempt + 1}/{max_retries})...")
                    time.sleep(wait_time)
                    continue

                response.raise_for_status()

                # 解析响应
                res_json = response.json()
                if "message" in res_json:
                    content = res_json["message"]["content"].strip()
                else:
                    content = res_json.get("response", "").strip()

                return content.replace("请输出摘要：", "").replace("摘要：", "").strip()

            except requests.exceptions.Timeout:
                print(f"    └── API 请求超时，正在重试 (尝试 {attempt + 1}/{max_retries})...")
                time.sleep(2)
            except Exception as e:
                print(f"    └── API 调用异常: {e} (尝试 {attempt + 1}/{max_retries})...")
                time.sleep(3)

        return f"Error: API 调用失败，已达到最大重试次数 ({max_retries}次)。"

    def _chunk_text(self, text, chunk_size=4000):
        return [text[i:i + chunk_size] for i in range(0, len(text), chunk_size)]

    def summarize(self, text):
        """1. 摘要 (采用级联摘要生成策略)"""
        if len(text) > 5000:
            chunks = self._chunk_text(text)
            partial_summaries = [self._call_llm(f"请简要概括以下片段(200字内):\n{c}") for c in chunks]
            combined_text = "\n".join(partial_summaries)
            prompt = f"请将以下多个片段的摘要汇总为一个连贯的最终摘要(300字内)：\n{combined_text}"
        else:
            prompt = f"请对以下文本进行简明扼要的摘要（300字以内）：\n{text}"
        return self._call_llm(prompt)

    def analyze_sentiment(self, text):
        """
        2. 情感分析 (Prompt 迭代：引入 Few-shot 引导)
        展示从 Zero-shot 到 Few-shot 的演进，规范输出格式。
        """
        safe_text = text[:5000]
        prompt = f"""
        请分析以下文本的情感倾向。
        严格按以下 Markdown 格式输出。

        【示例 1】
        输入："这门课程的内容非常详实，老师讲得也很有趣，完全超出了我的预期！"
        输出：
        - **情感倾向**：正面
        - **情感强度**：5分
        - **简要理由**：表达了强烈的赞美和极高的满意度。

        【示例 2】
        输入："今天的数据集有些缺失，需要花时间清洗，不过总体能跑通。"
        输出：
        - **情感倾向**：中性
        - **情感强度**：3分
        - **简要理由**：陈述客观遇到的问题，但情绪平稳，无明显负面情绪。

        【当前任务】
        输入："{safe_text}"
        输出：
        """
        return self._call_llm(prompt)

    def extract_keywords(self, text):
        """
        3. 关键词提取 (Prompt 迭代：引入 Few-shot 引导)
        """
        safe_text = text[:5000]
        prompt = f"""
        请提取 5-8 个核心关键词，用逗号分隔，直接输出词汇即可。

        【示例】
        输入："大语言模型（LLM）在自然语言处理任务中展现了惊人的涌现能力，如上下文学习和思维链推理。"
        输出：大语言模型, 自然语言处理, 涌现能力, 上下文学习, 思维链

        【当前任务】
        输入："{safe_text}"
        输出：
        """
        return self._call_llm(prompt)

    def compare_multiple_texts(self, files_dict):
        """
        4. 进阶功能升级：支持 N 篇文档的综合对比分析 (多文档对比)
        - 引入动态上下文分配算法 (Dynamic Context Allocation)
        """
        num_files = len(files_dict)
        if num_files < 2:
            return "Error: 对比分析至少需要 2 份文件。"

        # 【核心算法】动态截断策略：大模型最大安全上下文约为 24000 字符。
        # 根据文件数量动态均分上下文限额，确保永远不爆显存。
        max_chars_per_file = 24000 // num_files

        docs_content = ""
        for name, text in files_dict.items():
            # 动态截取前 N 个字符
            safe_text = text[:max_chars_per_file]
            docs_content += f"\n\n====================\n【文档：{name}】\n{safe_text}"

        # Prompt 迭代：支持 N 份文档的动态指令
        prompt = f"""
        请对以下 {num_files} 篇文档进行深度的综合对比分析。

        【思考过程指导 (Chain-of-Thought)】
        请遵循以下步骤进行推理：
        1. 分别概括每篇文档的核心主旨。
        2. 寻找所有文档的共性（他们在讨论什么共同话题？）。
        3. 深入对比它们在观点、侧重点或情感倾向上存在的主要差异。

        【输出格式限制】
        请不要输出你的思考过程，直接按以下严格的 Markdown 格式输出最终报告：

        ### 📑 1. 核心主旨概括
        (请逐一列出这 {num_files} 篇文档的一句话概括)

        ### 🤝 2. 共同点提取
        [列出核心共性]

        ### ⚖️ 3. 核心差异对比
        - **维度一 ([提取一个差异维度])**：文档A表现为...，文档B表现为...，文档C...
        - **维度二 ([提取第二个差异维度])**：各文档在...上的不同强调...

        ---
        【以下为多篇文档的文本内容】:
        {docs_content}
        """
        return self._call_llm(prompt)

    def generate_full_report_parallel(self, text):
        """并发生成报告"""
        with concurrent.futures.ThreadPoolExecutor(max_workers=3) as executor:
            future_summary = executor.submit(self.summarize, text)
            future_sentiment = executor.submit(self.analyze_sentiment, text)
            future_keywords = executor.submit(self.extract_keywords, text)

            summary = future_summary.result()
            sentiment = future_sentiment.result()
            keywords = future_keywords.result()

        return summary, sentiment, keywords

In [ ]:
import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output
import time
import io
import os
import PyPDF2


#1. 初始化后端
analyzer = TextAnalyzer(model_name="qwen3-coder:30b")

#2. 构建界面组件
header = widgets.HTML("<h2> 文本智能分析流水线 </h2>")

mode_selector = widgets.RadioButtons(
    options=['常规批量流水线 (提取摘要/情感/关键词)', '多文档综合对比分析 (支持2篇及以上)'],
    description='运行模式:',
    layout=widgets.Layout(width='100%', margin='0 0 15px 0')
)

# Tab 1: 文本输入
input_text_area = widgets.Textarea(placeholder='在此粘贴长文本...', layout=widgets.Layout(width='98%', height='150px'))
tab1 = widgets.VBox([widgets.Label("直接输入(单文本):"), input_text_area])

# Tab 2: 浏览器上传
file_uploader = widgets.FileUpload(accept='.txt, .pdf', multiple=True, description='浏览器上传')
upload_preview_out = widgets.Output(layout={'border': '1px dashed #ccc', 'padding': '10px', 'margin-top': '10px'})
tab2 = widgets.VBox([
    widgets.Label("注意: 受限于 Jupyter 内存限制，大文件或多份 PDF 可能导致浏览器上传失败。"),
    file_uploader,
    upload_preview_out
])

# Tab 3: 本地文件夹直读
folder_path_input = widgets.Text(
    placeholder='例如: ./my_pdfs 或者 C:/data/papers',
    description='文件夹路径:',
    layout=widgets.Layout(width='80%')
)
btn_scan_folder = widgets.Button(description='扫描此文件夹', button_style='info')
folder_preview_out = widgets.Output(layout={'border': '1px dashed #ccc', 'padding': '10px', 'margin-top': '10px'})
tab3 = widgets.VBox([
    widgets.Label("直接从本地/服务器读取文件夹内的所有 TXT/PDF。"),
    widgets.HBox([folder_path_input, btn_scan_folder]),
    folder_preview_out
])

# 组装 Tabs
tabs = widgets.Tab(children=[tab1, tab2, tab3])
tabs.set_title(0, '📝 文本粘贴')
tabs.set_title(1, '📂 浏览器上传 (小文件)')
tabs.set_title(2, '📁 文件夹直读 (推荐/大批量)')

# 控制区
btn_run = widgets.Button(description='启动流水线', button_style='primary',
                         layout=widgets.Layout(width='100%', height='50px', margin='15px 0 0 0'), icon='bolt')
progress = widgets.FloatProgress(value=0.0, min=0.0, max=100.0, description='准备就绪', bar_style='info',
                                 layout=widgets.Layout(width='100%', visibility='hidden'))
out = widgets.Output(layout={'border': '1px solid #ddd', 'padding': '15px', 'margin-top': '10px'})


#3. 核心解析逻辑
def extract_text_from_bytes(name, content_bytes):
    """通用的字节流提取函数 (支持 TXT 和 PDF)"""
    if name.lower().endswith('.pdf'):
        try:
            pdf_reader = PyPDF2.PdfReader(io.BytesIO(content_bytes))
            if pdf_reader.is_encrypted: return "[解析失败] PDF 已加密"
            extracted_text = ""
            for page in pdf_reader.pages:
                try:
                    text = page.extract_text()
                    if text: extracted_text += text + "\n"
                except:
                    pass
            if not extracted_text.strip(): return "[解析警告] 提取为空，可能是纯图片扫描件"
            return extracted_text
        except Exception as e:
            return f"[解析崩溃] PDF 损坏 ({e})"
    else:
        try:
            return content_bytes.decode("utf-8")
        except Exception as e:
            return f"[编码错误] TXT 需为 UTF-8 ({e})"


def render_dashboard(files_dict, output_widget):
    """渲染状态仪表盘"""
    with output_widget:
        clear_output()
        if not files_dict:
            display(Markdown("未检测到有效文件"))
            return

        table_md = "| 文件名 | 状态 | 提取字符数 | 预估 Token (约) |\n| :--- | :---: | :---: | :---: |\n"
        total_chars, total_tokens, valid_files = 0, 0, 0

        for name, text in files_dict.items():
            if text.startswith("⚠️"):
                table_md += f"| `{name}` | 异常 | 0 | 0 |\n  > *{text}*\n"
            else:
                chars = len(text)
                tokens = int(chars * 0.7)
                total_chars += chars;
                total_tokens += tokens;
                valid_files += 1
                table_md += f"| `{name}` | 成功 | {chars:,} 字 | ~{tokens:,} |\n"

        summary = f"### 🎛️ 实时文件仪表盘\n- **成功提取**：{valid_files} 份文件\n- **总文本量**：{total_chars:,} 字符 (预估将消耗 {total_tokens:,} Token)\n\n"
        if total_tokens > 24000:
            summary += "> **智能提醒**：当前总 Token 较高，将自动触发长文本降维策略。\n\n"
        display(Markdown(summary + table_md))


#4. 监听器绑定
# 监听 Tab2: 浏览器上传
def on_file_upload_change(change):
    if not file_uploader.value: return
    with upload_preview_out:
        print("⏳ 正在扫描解析文档，请稍候...")
    files_dict = {}
    items = file_uploader.value if isinstance(file_uploader.value, tuple) else file_uploader.value.values()
    for info in items:
        name = info['name'] if isinstance(info, dict) else info.name
        raw_data = info['content'] if isinstance(info, dict) else info.content
        b_data = raw_data.tobytes() if isinstance(raw_data, memoryview) else bytes(raw_data)
        files_dict[name] = extract_text_from_bytes(name, b_data)
    render_dashboard(files_dict, upload_preview_out)


file_uploader.observe(on_file_upload_change, names='value')

# 监听 Tab3: 扫描本地文件夹
global_folder_files = {}  # 存储文件夹读取到的内容


def on_scan_folder_click(b):
    folder_path = folder_path_input.value.strip()
    global global_folder_files
    global_folder_files.clear()

    with folder_preview_out:
        clear_output()
        if not folder_path or not os.path.exists(folder_path):
            print(f"找不到该文件夹: {folder_path} (请确保路径正确，如果是当前目录可用 ./xxx )")
            return
        print(f"⏳ 正在读取硬盘文件夹: {folder_path} ...")

    for filename in os.listdir(folder_path):
        if filename.lower().endswith('.pdf') or filename.lower().endswith('.txt'):
            file_path = os.path.join(folder_path, filename)
            try:
                with open(file_path, 'rb') as f:
                    content_bytes = f.read()
                global_folder_files[filename] = extract_text_from_bytes(filename, content_bytes)
            except Exception as e:
                global_folder_files[filename] = f"无法读取文件 ({e})"

    render_dashboard(global_folder_files, folder_preview_out)


btn_scan_folder.on_click(on_scan_folder_click)


#5. 核心启动逻辑
def on_click_analysis(b):
    out.clear_output()
    btn_run.disabled = True
    progress.layout.visibility = 'visible'
    progress.value = 5
    progress.bar_style = 'info'

    current_tab = tabs.selected_index
    start_time = time.time()
    texts_to_process = {}

    try:
        # 获取待处理的文本源
        if current_tab == 0:
            if not input_text_area.value.strip(): raise Exception("文本内容为空！")
            texts_to_process["手动输入的文本"] = input_text_area.value
        elif current_tab == 1:
            if not file_uploader.value: raise Exception("请先上传文件！(或尝试切换到第3个选项卡读取文件夹)")
            items = file_uploader.value if isinstance(file_uploader.value, tuple) else file_uploader.value.values()
            for info in items:
                name = info['name'] if isinstance(info, dict) else info.name
                raw_data = info['content'] if isinstance(info, dict) else info.content
                b_data = raw_data.tobytes() if isinstance(raw_data, memoryview) else bytes(raw_data)
                texts_to_process[name] = extract_text_from_bytes(name, b_data)
        elif current_tab == 2:
            if not global_folder_files: raise Exception("请先输入路径并点击【扫描文件夹】！")
            texts_to_process = global_folder_files.copy()

        # 过滤掉带有 ⚠️ 警告的无效文件
        valid_files = {k: v for k, v in texts_to_process.items() if not v.startswith("⚠️")}

        if '对比分析' in mode_selector.value:
            if len(valid_files) < 2: raise Exception(f"对比分析需至少 2 份有效文件，当前有效 {len(valid_files)} 份！")
            progress.value = 20
            progress.description = '正在对比...'
            with out:
                print(f"⏳ 正在对 {len(valid_files)} 份文档进行对比，请稍候...")
            compare_result = analyzer.compare_multiple_texts(valid_files)

            progress.value = 100
            with out:
                clear_output()
                display(Markdown(
                    f"# 多文档综合对比报告\n> ⏱️ **总耗时**: {time.time() - start_time:.2f}s\n\n{compare_result}"))

        else:
            if not valid_files: raise Exception("没有检测到有效文件，请检查仪表盘报错。")
            total_files = len(valid_files)
            all_reports = []

            for idx, (filename, text_content) in enumerate(valid_files.items()):
                progress.description = f'处理中 ({idx + 1}/{total_files})'
                with out: print(f"⏳ 正在处理[{idx + 1}/{total_files}]: {filename} ...")

                summary, sentiment, keywords = analyzer.generate_full_report_parallel(text_content)
                all_reports.append(
                    f"## 📄 {filename}\n- **🔑 摘要**: {summary}\n- **🎭 情感**: \n{sentiment}\n- **🏷️ 关键词**: {keywords}\n---")
                progress.value = 10 + (90 / total_files) * (idx + 1)

            progress.value = 100
            with out:
                clear_output()
                display(Markdown(
                    f"# 📊 批量分析完成\n> 📑 **共成功处理**: {total_files} 份\n> ⏱️ **总耗时**: {time.time() - start_time:.2f} 秒\n\n" + "\n".join(
                        all_reports)))

        progress.description = '完成！'
        progress.bar_style = 'success'

    except Exception as e:
        with out:
            display(Markdown(f"**发生错误**: {e}"))
        progress.bar_style = 'danger'
    finally:
        btn_run.disabled = False


btn_run.on_click(on_click_analysis)

# --- 5. 显示界面 ---
ui = widgets.VBox([header, mode_selector, tabs, btn_run, progress, out])
display(ui)